# QASC Quantum-AI Galaxy Classifier/Retriever




In [ ]:
!pip install -q qiskit qiskit-aer pandas matplotlib torch torchvision huggingface_hub pillow scikit-learn astroNN h5py

## 1. Core project code
Run this cell once. It defines the classifier, catalog, Grover search, and validation functions.


In [ ]:

from __future__ import annotations

import os
import math
from dataclasses import dataclass
from typing import Dict, Iterable, List, Optional, Tuple, Union

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from huggingface_hub import hf_hub_download

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

try:
    from google.colab import files  # type: ignore
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

try:
    from sklearn.metrics import classification_report, confusion_matrix
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False



# 1. Galaxy labels and binary tag system


GALAXY10_LABELS: Dict[int, str] = {
    0: "Disk, Face-on, No Spiral",
    1: "Smooth, Completely round",
    2: "Smooth, in-between round",
    3: "Smooth, Cigar shaped",
    4: "Disk, Edge-on, Rounded Bulge",
    5: "Disk, Edge-on, Boxy Bulge",
    6: "Disk, Edge-on, No Bulge",
    7: "Disk, Face-on, Tight Spiral",
    8: "Disk, Face-on, Medium Spiral",
    9: "Disk, Face-on, Loose Spiral",
}

# Five-class project tag structure.
# Galaxy10 SDSS validates only Elliptical, Spiral, and Lenticular in this prototype.
GALAXY_TAGS: Dict[str, str] = {
    "Elliptical": "000",
    "Spiral": "001",
    "Lenticular": "010",
    "Irregular": "011",
    "Merging": "111",
}

# Project mapping used for this paper.
# Edge-on disk classes are treated as Spiral/Disk rather than Lenticular because
# they contain clear disk morphology in Galaxy10.
GALAXY10_TO_PROJECT_CLASS: Dict[int, str] = {
    0: "Lenticular",
    1: "Elliptical",
    2: "Elliptical",
    3: "Elliptical",
    4: "Spiral",
    5: "Spiral",
    6: "Spiral",
    7: "Spiral",
    8: "Spiral",
    9: "Spiral",
}

VALIDATED_PROJECT_CLASSES = ["Elliptical", "Lenticular", "Spiral"]



# 2. Pretrained Galaxy10 classifier


class PremadeGalaxy10Classifier:
    """Loads a pretrained Galaxy10 classifier and predicts galaxy morphology."""

    def __init__(self, model_name: str = "DenseNet121", transform_style: str = "native_raw_0_1"):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_name = model_name

        print("Using device:", self.device)
        print(f"Loading premade Galaxy10 SDSS classifier: {model_name}")

        self.model = self._load_model(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.set_transform(transform_style)

        print("Classifier loaded.")

    def set_transform(self, style: str = "native_raw_0_1") -> None:
        """Sets image preprocessing. native_raw_0_1 performed best for Galaxy10 validation."""
        if style == "native_raw_0_1":
            self.transform = transforms.Compose([
                transforms.Resize((69, 69)),
                transforms.ToTensor(),
            ])
        elif style == "raw_0_1_224":
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
            ])
        elif style == "imagenet_224":
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225],
                ),
            ])
        else:
            raise ValueError("Choose: native_raw_0_1, raw_0_1_224, or imagenet_224")

        self.transform_style = style

    def _checkpoint_state_dict(self, repo_id: str, filename: str):
        weights_path = hf_hub_download(repo_id=repo_id, filename=filename)
        checkpoint = torch.load(weights_path, map_location=self.device)

        if not isinstance(checkpoint, dict):
            return None, checkpoint

        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        else:
            state_dict = checkpoint

        state_dict = {key.replace("module.", ""): value for key, value in state_dict.items()}
        return state_dict, None

    def _load_model(self, model_name: str) -> nn.Module:
        repo_id = "enyasantos/galaxy-classification-v01"

        if model_name == "DenseNet121":
            filename = "models/DenseNet121.pth"
            state_dict, full_model = self._checkpoint_state_dict(repo_id, filename)
            if full_model is not None:
                return full_model

            model = models.densenet121(weights=None)
            if "classifier.0.weight" in state_dict and "classifier.6.weight" in state_dict:
                w0 = state_dict["classifier.0.weight"]
                w3 = state_dict["classifier.3.weight"]
                w6 = state_dict["classifier.6.weight"]
                model.classifier = nn.Sequential(
                    nn.Linear(w0.shape[1], w0.shape[0]),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.5),
                    nn.Linear(w3.shape[1], w3.shape[0]),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.5),
                    nn.Linear(w6.shape[1], w6.shape[0]),
                )
            else:
                model.classifier = nn.Linear(model.classifier.in_features, 10)

        elif model_name == "ResNet50":
            filename = "models/ResNet50.pth"
            state_dict, full_model = self._checkpoint_state_dict(repo_id, filename)
            if full_model is not None:
                return full_model

            model = models.resnet50(weights=None)
            if "fc.0.weight" in state_dict and "fc.6.weight" in state_dict:
                w0 = state_dict["fc.0.weight"]
                w3 = state_dict["fc.3.weight"]
                w6 = state_dict["fc.6.weight"]
                model.fc = nn.Sequential(
                    nn.Linear(w0.shape[1], w0.shape[0]),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.5),
                    nn.Linear(w3.shape[1], w3.shape[0]),
                    nn.ReLU(inplace=True),
                    nn.Dropout(p=0.5),
                    nn.Linear(w6.shape[1], w6.shape[0]),
                )
            else:
                model.fc = nn.Linear(model.fc.in_features, 10)

        else:
            raise ValueError("Choose one of: DenseNet121 or ResNet50")

        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print("Model loading diagnostic")
        print("------------------------")
        print("Missing keys:", len(missing))
        print("Unexpected keys:", len(unexpected))
        if len(missing) == 0 and len(unexpected) == 0:
            print("All checkpoint weights loaded cleanly.")
        else:
            print("First missing keys:", missing[:10])
            print("First unexpected keys:", unexpected[:10])

        return model

    def predict(self, image_path: str) -> Dict[str, Union[int, str, float]]:
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"Image path not found: {image_path}")

        image = Image.open(image_path).convert("RGB")
        x = self.transform(image).unsqueeze(0).to(self.device)

        with torch.no_grad():
            logits = self.model(x)
            probabilities = F.softmax(logits, dim=1)[0]

        class_id = int(torch.argmax(probabilities).item())
        confidence = float(probabilities[class_id].item())
        galaxy_type = GALAXY10_TO_PROJECT_CLASS[class_id]

        return {
            "galaxy10_class_id": class_id,
            "galaxy10_label": GALAXY10_LABELS[class_id],
            "galaxy_type": galaxy_type,
            "binary_tag": GALAXY_TAGS[galaxy_type],
            "confidence": confidence,
        }



# 3. Catalog storage

class GalaxyCatalog:
    """Stores galaxy rows, predicted morphology tags, and optional true labels."""

    columns = [
        "row_number",
        "sdss_name",
        "image_path",
        "galaxy10_class_id",
        "galaxy10_label",
        "galaxy_type",
        "binary_tag",
        "confidence",
        "true_galaxy10_class_id",
        "true_galaxy_type",
        "is_correct_project_class",
    ]

    def __init__(self):
        self.catalog = pd.DataFrame(columns=self.columns)
        self.next_row_number = 0

    def add_galaxy(
        self,
        sdss_name: str,
        image_path: str,
        galaxy10_class_id: int,
        galaxy10_label: str,
        galaxy_type: str,
        confidence: float,
        true_galaxy10_class_id: Optional[int] = None,
        true_galaxy_type: Optional[str] = None,
    ) -> None:
        if galaxy_type not in GALAXY_TAGS:
            raise ValueError(f"Invalid predicted galaxy type: {galaxy_type}")

        if true_galaxy_type is None and true_galaxy10_class_id is not None:
            true_galaxy_type = GALAXY10_TO_PROJECT_CLASS[int(true_galaxy10_class_id)]

        is_correct = None
        if true_galaxy_type is not None:
            is_correct = galaxy_type == true_galaxy_type

        self.catalog.loc[len(self.catalog)] = {
            "row_number": self.next_row_number,
            "sdss_name": sdss_name,
            "image_path": image_path,
            "galaxy10_class_id": int(galaxy10_class_id),
            "galaxy10_label": galaxy10_label,
            "galaxy_type": galaxy_type,
            "binary_tag": GALAXY_TAGS[galaxy_type],
            "confidence": float(confidence),
            "true_galaxy10_class_id": true_galaxy10_class_id,
            "true_galaxy_type": true_galaxy_type,
            "is_correct_project_class": is_correct,
        }
        self.next_row_number += 1

    def display_catalog(self) -> pd.DataFrame:
        return self.catalog.copy()

    def get_matching_rows(self, galaxy_type: str, use_true_labels: bool = False) -> List[int]:
        if galaxy_type not in GALAXY_TAGS:
            raise ValueError(f"Invalid galaxy type: {galaxy_type}. Choose from {list(GALAXY_TAGS)}")

        if use_true_labels:
            matches = self.catalog[self.catalog["true_galaxy_type"] == galaxy_type]
        else:
            target_tag = GALAXY_TAGS[galaxy_type]
            matches = self.catalog[self.catalog["binary_tag"] == target_tag]

        return matches["row_number"].astype(int).tolist()

    def get_row(self, row_number: int) -> Optional[pd.Series]:
        result = self.catalog[self.catalog["row_number"] == int(row_number)]
        return None if len(result) == 0 else result.iloc[0]

    def save_to_csv(self, filename: str = "classified_galaxy_catalog.csv") -> None:
        self.catalog.to_csv(filename, index=False)
        print(f"Catalog saved to {filename}")



# 4. Image and dataset helpers


def add_image_to_catalog(
    catalog: GalaxyCatalog,
    classifier: PremadeGalaxy10Classifier,
    sdss_name: str,
    image_path: str,
    true_galaxy10_class_id: Optional[int] = None,
    true_galaxy_type: Optional[str] = None,
    verbose: bool = False,
) -> Dict[str, Union[int, str, float]]:
    prediction = classifier.predict(image_path)

    catalog.add_galaxy(
        sdss_name=sdss_name,
        image_path=image_path,
        galaxy10_class_id=int(prediction["galaxy10_class_id"]),
        galaxy10_label=str(prediction["galaxy10_label"]),
        galaxy_type=str(prediction["galaxy_type"]),
        confidence=float(prediction["confidence"]),
        true_galaxy10_class_id=true_galaxy10_class_id,
        true_galaxy_type=true_galaxy_type,
    )

    if verbose:
        print(f"Added {sdss_name}: predicted {prediction['galaxy_type']} ({prediction['confidence']:.4f})")

    return prediction


def build_catalog_from_csv(
    csv_path: str,
    classifier: PremadeGalaxy10Classifier,
    image_root: str = "",
    limit: Optional[int] = None,
) -> GalaxyCatalog:
    """Builds a classified catalog from a CSV containing image_path and true labels."""
    df = pd.read_csv(csv_path)
    if "image_path" not in df.columns:
        raise ValueError("CSV must contain an 'image_path' column.")
    if limit is not None:
        df = df.head(limit)

    catalog = GalaxyCatalog()
    for index, row in df.iterrows():
        image_path = str(row["image_path"])
        if image_root:
            image_path = os.path.join(image_root, image_path)

        sdss_name = str(row["sdss_name"]) if "sdss_name" in df.columns else f"galaxy_{index}"
        true_galaxy10_class_id = None
        true_galaxy_type = None

        if "true_galaxy10_class_id" in df.columns and pd.notna(row["true_galaxy10_class_id"]):
            true_galaxy10_class_id = int(row["true_galaxy10_class_id"])
        if "true_galaxy_type" in df.columns and pd.notna(row["true_galaxy_type"]):
            true_galaxy_type = str(row["true_galaxy_type"])

        add_image_to_catalog(
            catalog=catalog,
            classifier=classifier,
            sdss_name=sdss_name,
            image_path=image_path,
            true_galaxy10_class_id=true_galaxy10_class_id,
            true_galaxy_type=true_galaxy_type,
            verbose=False,
        )

    return catalog


def create_balanced_galaxy10_subset(
    images,
    validation_df: pd.DataFrame,
    samples_per_class: int = 30,
    output_dir: str = "galaxy10_validation_images",
    csv_path: str = "balanced_galaxy10_validation.csv",
    random_state: int = 42,
) -> pd.DataFrame:
    """Creates a balanced subset of Galaxy10 images and writes a CSV for validation."""
    import numpy as np

    os.makedirs(output_dir, exist_ok=True)

    balanced_df = (
        validation_df
        .groupby("true_galaxy_type", group_keys=False)
        .apply(lambda x: x.sample(n=samples_per_class, random_state=random_state))
        .reset_index(drop=True)
    )

    image_paths = []
    for _, row in balanced_df.iterrows():
        dataset_index = int(row["dataset_index"])
        true_type = str(row["true_galaxy_type"])
        true_class_id = int(row["true_galaxy10_class_id"])
        img_array = images[dataset_index]
        if img_array.dtype != np.uint8:
            img_array = np.clip(img_array, 0, 255).astype(np.uint8)

        filename = f"galaxy10_{dataset_index}_class{true_class_id}_{true_type}.png"
        path = os.path.join(output_dir, filename)
        Image.fromarray(img_array).save(path)
        image_paths.append(path)

    balanced_df["image_path"] = image_paths
    balanced_df["sdss_name"] = balanced_df["dataset_index"].apply(lambda x: f"Galaxy10_Index_{x}")
    balanced_df = balanced_df[[
        "image_path",
        "sdss_name",
        "dataset_index",
        "true_galaxy10_class_id",
        "true_galaxy10_label",
        "true_galaxy_type",
    ]]
    balanced_df.to_csv(csv_path, index=False)
    return balanced_df



# 5. Binary row encoding and Grover search


def bits_needed_for_catalog(total_rows: int) -> int:
    return 1 if total_rows <= 1 else math.ceil(math.log2(total_rows))


def row_number_to_binary(row_number: int, total_rows: int) -> str:
    return format(int(row_number), f"0{bits_needed_for_catalog(total_rows)}b")


def binary_to_row_number(binary_string: str) -> int:
    return int(binary_string, 2)


def apply_phase_flip_for_state(qc: QuantumCircuit, qubits: List[int], target_bitstring: str) -> None:
    reversed_bits = target_bitstring[::-1]
    for i, bit in enumerate(reversed_bits):
        if bit == "0":
            qc.x(qubits[i])

    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1], qubits[-1])
        qc.h(qubits[-1])

    for i, bit in enumerate(reversed_bits):
        if bit == "0":
            qc.x(qubits[i])


def apply_oracle(qc: QuantumCircuit, qubits: List[int], marked_states: Iterable[str]) -> None:
    for state in marked_states:
        apply_phase_flip_for_state(qc, qubits, state)


def apply_diffuser(qc: QuantumCircuit, qubits: List[int]) -> None:
    qc.h(qubits)
    qc.x(qubits)
    if len(qubits) == 1:
        qc.z(qubits[0])
    else:
        qc.h(qubits[-1])
        qc.mcx(qubits[:-1], qubits[-1])
        qc.h(qubits[-1])
    qc.x(qubits)
    qc.h(qubits)


def recommended_grover_iterations(search_space_size: int, num_solutions: int) -> int:
    """Angle-based Grover iteration estimate to avoid small-N over-rotation."""
    if num_solutions <= 0:
        raise ValueError("num_solutions must be positive.")
    if num_solutions >= search_space_size:
        return 0
    theta = math.asin(math.sqrt(num_solutions / search_space_size))
    return max(0, round((math.pi / (4 * theta)) - 0.5))


def build_grover_circuit(
    total_rows: int,
    matching_rows: List[int],
    iterations: Optional[int] = None,
) -> Tuple[QuantumCircuit, List[str], int, int]:
    if total_rows <= 0:
        raise ValueError("Catalog must contain at least one row.")
    if len(matching_rows) == 0:
        raise ValueError("No matching rows found for this galaxy type.")

    num_qubits = bits_needed_for_catalog(total_rows)
    search_space_size = 2 ** num_qubits
    marked_states = [row_number_to_binary(row, total_rows) for row in matching_rows]

    if iterations is None:
        iterations = recommended_grover_iterations(search_space_size, len(marked_states))

    qc = QuantumCircuit(num_qubits, num_qubits)
    qubits = list(range(num_qubits))
    qc.h(qubits)

    for _ in range(iterations):
        apply_oracle(qc, qubits, marked_states)
        apply_diffuser(qc, qubits)

    qc.measure(qubits, qubits)
    return qc, marked_states, iterations, search_space_size


@dataclass
class GroverSearchResult:
    requested_type: str
    returned_row_number: Optional[int]
    returned_galaxy: Optional[pd.Series]
    counts: Dict[str, int]
    valid_row_counts: Dict[int, int]
    matching_rows: List[int]
    marked_states: List[str]
    iterations: int
    total_rows: int
    search_space_size: int
    shots: int
    success_probability: float
    random_baseline_probability: float
    amplification_factor: Optional[float]
    invalid_probability: float


def grover_search_by_galaxy_type(
    catalog: GalaxyCatalog,
    galaxy_type: str,
    shots: int = 1024,
    use_true_labels_for_oracle: bool = False,
    iterations: Optional[int] = None,
) -> GroverSearchResult:
    total_rows = len(catalog.display_catalog())
    matching_rows = catalog.get_matching_rows(galaxy_type, use_true_labels=use_true_labels_for_oracle)

    qc, marked_states, used_iterations, search_space_size = build_grover_circuit(
        total_rows=total_rows,
        matching_rows=matching_rows,
        iterations=iterations,
    )

    simulator = AerSimulator()
    compiled_circuit = transpile(qc, simulator)
    result = simulator.run(compiled_circuit, shots=shots).result()
    counts = dict(result.get_counts())

    valid_row_counts: Dict[int, int] = {}
    invalid_counts = 0
    for bitstring, count in counts.items():
        row_number = binary_to_row_number(bitstring)
        if row_number >= total_rows:
            invalid_counts += count
        else:
            valid_row_counts[row_number] = valid_row_counts.get(row_number, 0) + count

    marked_counts = {row: count for row, count in valid_row_counts.items() if row in matching_rows}
    returned_row_number = None
    if marked_counts:
        returned_row_number = max(marked_counts, key=marked_counts.get)
    elif valid_row_counts:
        returned_row_number = max(valid_row_counts, key=valid_row_counts.get)

    success_probability = sum(marked_counts.values()) / shots
    random_baseline_probability = len(matching_rows) / search_space_size
    invalid_probability = invalid_counts / shots
    amplification_factor = success_probability / random_baseline_probability if random_baseline_probability > 0 else None

    return GroverSearchResult(
        requested_type=galaxy_type,
        returned_row_number=returned_row_number,
        returned_galaxy=None if returned_row_number is None else catalog.get_row(returned_row_number),
        counts=counts,
        valid_row_counts=valid_row_counts,
        matching_rows=matching_rows,
        marked_states=marked_states,
        iterations=used_iterations,
        total_rows=total_rows,
        search_space_size=search_space_size,
        shots=shots,
        success_probability=success_probability,
        random_baseline_probability=random_baseline_probability,
        amplification_factor=amplification_factor,
        invalid_probability=invalid_probability,
    )


# 6. Validation metrics

def classifier_validation_metrics(catalog: GalaxyCatalog) -> Dict[str, object]:
    df = catalog.display_catalog()
    df = df[df["true_galaxy_type"].notna()].copy()
    if len(df) == 0:
        raise ValueError("No true labels found in catalog.")

    y_true = df["true_galaxy_type"].astype(str).tolist()
    y_pred = df["galaxy_type"].astype(str).tolist()
    labels = VALIDATED_PROJECT_CLASSES
    accuracy = sum(t == p for t, p in zip(y_true, y_pred)) / len(y_true)

    output: Dict[str, object] = {"num_labeled_images": len(df), "accuracy": accuracy, "labels": labels}
    print("AI Classifier Validation")
    print("------------------------")
    print("Number of labeled images:", len(df))
    print("Accuracy:", round(accuracy, 4))

    if SKLEARN_AVAILABLE:
        report_dict = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
        report_text = classification_report(y_true, y_pred, labels=labels, zero_division=0)
        cm = confusion_matrix(y_true, y_pred, labels=labels)
        cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels], columns=[f"pred_{x}" for x in labels])
        print("\nClassification report:")
        print(report_text)
        print("Confusion matrix:")
        display(cm_df)
        output["classification_report"] = report_dict
        output["classification_report_df"] = pd.DataFrame(report_dict).transpose()
        output["confusion_matrix"] = cm_df

    return output


def grover_validation_metrics(
    catalog: GalaxyCatalog,
    galaxy_types: Optional[List[str]] = None,
    shots: int = 1024,
    trials_per_type: int = 5,
    use_true_labels_for_oracle: bool = False,
) -> pd.DataFrame:
    if galaxy_types is None:
        galaxy_types = VALIDATED_PROJECT_CLASSES

    rows = []
    for galaxy_type in galaxy_types:
        matching_rows = catalog.get_matching_rows(galaxy_type, use_true_labels=use_true_labels_for_oracle)
        if len(matching_rows) == 0:
            print(f"Skipping {galaxy_type}: no matching rows.")
            continue

        for trial in range(1, trials_per_type + 1):
            result = grover_search_by_galaxy_type(
                catalog=catalog,
                galaxy_type=galaxy_type,
                shots=shots,
                use_true_labels_for_oracle=use_true_labels_for_oracle,
            )
            rows.append({
                "galaxy_type": galaxy_type,
                "trial": trial,
                "matching_rows": len(result.matching_rows),
                "search_space_size": result.search_space_size,
                "iterations": result.iterations,
                "shots": result.shots,
                "random_baseline_probability": result.random_baseline_probability,
                "success_probability": result.success_probability,
                "amplification_factor": result.amplification_factor,
                "invalid_probability": result.invalid_probability,
                "returned_row_number": result.returned_row_number,
            })

    out = pd.DataFrame(rows)
    if len(out) == 0:
        raise ValueError("No Grover validation results were produced.")
    return out


def summarize_grover_trials(grover_trials: pd.DataFrame) -> pd.DataFrame:
    summary = grover_trials.groupby("galaxy_type").agg(
        matching_rows=("matching_rows", "first"),
        search_space_size=("search_space_size", "first"),
        iterations=("iterations", "first"),
        random_baseline_probability=("random_baseline_probability", "first"),
        mean_success_probability=("success_probability", "mean"),
        std_success_probability=("success_probability", "std"),
        mean_amplification_factor=("amplification_factor", "mean"),
        mean_invalid_probability=("invalid_probability", "mean"),
    ).reset_index()
    display(summary)
    return summary


def end_to_end_validation_metrics(
    catalog: GalaxyCatalog,
    galaxy_types: Optional[List[str]] = None,
    shots: int = 1024,
    trials_per_type: int = 5,
) -> pd.DataFrame:
    """Measures full pipeline: predicted AI tag -> Grover retrieval -> returned galaxy correctness."""
    if galaxy_types is None:
        galaxy_types = VALIDATED_PROJECT_CLASSES

    rows = []
    for galaxy_type in galaxy_types:
        matching_rows = catalog.get_matching_rows(galaxy_type, use_true_labels=False)
        if len(matching_rows) == 0:
            print(f"Skipping {galaxy_type}: no AI-predicted matching rows.")
            continue

        for trial in range(1, trials_per_type + 1):
            result = grover_search_by_galaxy_type(
                catalog=catalog,
                galaxy_type=galaxy_type,
                shots=shots,
                use_true_labels_for_oracle=False,
            )

            returned_predicted_type = None
            predicted_type_success = False
            true_type_success = None
            if result.returned_galaxy is not None:
                returned_predicted_type = str(result.returned_galaxy["galaxy_type"])
                predicted_type_success = returned_predicted_type == galaxy_type
                true_type = result.returned_galaxy.get("true_galaxy_type", None)
                if pd.notna(true_type):
                    true_type_success = str(true_type) == galaxy_type

            rows.append({
                "requested_type": galaxy_type,
                "trial": trial,
                "returned_row_number": result.returned_row_number,
                "returned_predicted_type": returned_predicted_type,
                "predicted_type_success": predicted_type_success,
                "true_type_success": true_type_success,
                "grover_success_probability": result.success_probability,
                "random_baseline_probability": result.random_baseline_probability,
                "amplification_factor": result.amplification_factor,
                "invalid_probability": result.invalid_probability,
            })

    out = pd.DataFrame(rows)
    if len(out) == 0:
        raise ValueError("No end-to-end validation results were produced.")
    return out


def summarize_end_to_end_trials(e2e_trials: pd.DataFrame) -> pd.DataFrame:
    summary = e2e_trials.groupby("requested_type").agg(
        predicted_tag_success_rate=("predicted_type_success", "mean"),
        true_label_success_rate=("true_type_success", "mean"),
        mean_grover_success_probability=("grover_success_probability", "mean"),
        mean_random_baseline_probability=("random_baseline_probability", "mean"),
        mean_amplification_factor=("amplification_factor", "mean"),
        mean_invalid_probability=("invalid_probability", "mean"),
    ).reset_index()
    display(summary)
    return summary


def save_validation_outputs(
    catalog: Optional[GalaxyCatalog] = None,
    classifier_metrics: Optional[Dict[str, object]] = None,
    grover_trials: Optional[pd.DataFrame] = None,
    grover_summary: Optional[pd.DataFrame] = None,
    e2e_trials: Optional[pd.DataFrame] = None,
    e2e_summary: Optional[pd.DataFrame] = None,
) -> None:
    if catalog is not None:
        catalog.save_to_csv("classified_galaxy_catalog.csv")
    if classifier_metrics is not None:
        if "classification_report_df" in classifier_metrics:
            classifier_metrics["classification_report_df"].to_csv("ai_classification_report.csv")
            print("Saved ai_classification_report.csv")
        if "confusion_matrix" in classifier_metrics:
            classifier_metrics["confusion_matrix"].to_csv("ai_confusion_matrix.csv")
            print("Saved ai_confusion_matrix.csv")
    if grover_trials is not None:
        grover_trials.to_csv("grover_validation_trials.csv", index=False)
        print("Saved grover_validation_trials.csv")
    if grover_summary is not None:
        grover_summary.to_csv("grover_validation_summary.csv", index=False)
        print("Saved grover_validation_summary.csv")
    if e2e_trials is not None:
        e2e_trials.to_csv("end_to_end_validation_trials.csv", index=False)
        print("Saved end_to_end_validation_trials.csv")
    if e2e_summary is not None:
        e2e_summary.to_csv("end_to_end_validation_summary.csv", index=False)
        print("Saved end_to_end_validation_summary.csv")


def download_if_colab(filename: str) -> None:
    if IN_COLAB and files is not None and os.path.exists(filename):
        files.download(filename)
    else:
        print(f"File ready: {filename}")


## 2. Load Galaxy10 SDSS and create labeled dataframe
This loads the real labeled Galaxy10 SDSS dataset and maps the 10 Galaxy10 labels into your 3 validated project classes: Elliptical, Lenticular, and Spiral.


In [ ]:
import numpy as np
from astroNN.datasets import load_galaxy10sdss

images, labels = load_galaxy10sdss()
labels = labels.astype(int)

validation_df = pd.DataFrame({
    "dataset_index": np.arange(len(labels)),
    "true_galaxy10_class_id": labels,
})
validation_df["true_galaxy10_label"] = validation_df["true_galaxy10_class_id"].map(GALAXY10_LABELS)
validation_df["true_galaxy_type"] = validation_df["true_galaxy10_class_id"].map(GALAXY10_TO_PROJECT_CLASS)

print("Loaded Galaxy10 SDSS")
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Mapped project-class counts:")
print(validation_df["true_galaxy_type"].value_counts())
display(validation_df.head())

## 3. Create balanced validation subset
Start with 30 images per class. Increase `SAMPLES_PER_CLASS` later if you want stronger validation.


In [ ]:
SAMPLES_PER_CLASS = 30

balanced_df = create_balanced_galaxy10_subset(
    images=images,
    validation_df=validation_df,
    samples_per_class=SAMPLES_PER_CLASS,
    output_dir="galaxy10_validation_images",
    csv_path="balanced_galaxy10_validation.csv",
    random_state=42,
)

print("Balanced validation subset:")
print(balanced_df["true_galaxy_type"].value_counts())
print("Total validation images:", len(balanced_df))
display(balanced_df.head())

## 4. Build AI-tagged validation catalog
This uses the best preprocessing found during debugging: `DenseNet121` with `native_raw_0_1`.


In [ ]:
classifier = PremadeGalaxy10Classifier(
    model_name="DenseNet121",
    transform_style="native_raw_0_1",
)

validation_catalog = build_catalog_from_csv(
    csv_path="balanced_galaxy10_validation.csv",
    classifier=classifier,
)

catalog_df = validation_catalog.display_catalog()
print("Total rows in validation catalog:", len(catalog_df))
print("Predicted class counts:")
print(catalog_df["galaxy_type"].value_counts())
print("True class counts:")
print(catalog_df["true_galaxy_type"].value_counts())
display(catalog_df.head())

## 5. AI classifier validation
This produces the accuracy, precision, recall, F1-score, and confusion matrix for the AI stage.


In [ ]:
classifier_metrics = classifier_validation_metrics(validation_catalog)

## 6. Grover retrieval validation on the real AI-tagged catalog
This validates whether Grover amplifies the probability of rows matching the AI-generated binary tags.


In [ ]:
grover_trials_real = grover_validation_metrics(
    catalog=validation_catalog,
    galaxy_types=VALIDATED_PROJECT_CLASSES,
    shots=1024,
    trials_per_type=5,
    use_true_labels_for_oracle=False,
)

grover_summary_real = summarize_grover_trials(grover_trials_real)

## 7. End-to-end hybrid validation
This measures the full pipeline: image classification → binary tag → Grover retrieval → returned galaxy.

`predicted_tag_success_rate` checks whether Grover returned a row matching the AI tag.

`true_label_success_rate` checks whether the returned row also matches the official Galaxy10-derived label.


In [ ]:
e2e_trials = end_to_end_validation_metrics(
    catalog=validation_catalog,
    galaxy_types=VALIDATED_PROJECT_CLASSES,
    shots=1024,
    trials_per_type=5,
)

e2e_summary = summarize_end_to_end_trials(e2e_trials)

## 8. Save outputs for the paper
This saves clean CSV tables you can use in the results section.


In [ ]:
save_validation_outputs(
    catalog=validation_catalog,
    classifier_metrics=classifier_metrics,
    grover_trials=grover_trials_real,
    grover_summary=grover_summary_real,
    e2e_trials=e2e_trials,
    e2e_summary=e2e_summary,
)

for filename in [
    "classified_galaxy_catalog.csv",
    "ai_classification_report.csv",
    "ai_confusion_matrix.csv",
    "grover_validation_trials.csv",
    "grover_validation_summary.csv",
    "end_to_end_validation_trials.csv",
    "end_to_end_validation_summary.csv",
]:
    download_if_colab(filename)